In [1]:
import kaggle_environments as ke

[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO: Successfully loaded OpenSpiel environments: 17.
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_amazons
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_backgammon
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_checkers
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_chess
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_connect_four
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_dark_hex
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_gin_rummy
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_go
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_goofspiel
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_hearts
[kaggle_environments.envs.open_sp

In [2]:
from collections import namedtuple
import math

# Named tuples for agent convenience.
# Planets and fleets share a common [id, owner, x, y, ...] prefix.
Planet = namedtuple(
    "Planet", ["id", "owner", "x", "y", "radius", "ships", "production"]
)
Fleet = namedtuple(
    "Fleet", ["id", "owner", "x", "y", "angle", "from_planet_id", "ships"]
)

# Constants
BOARD_SIZE = 100.0
CENTER = BOARD_SIZE / 2.0
SUN_RADIUS = 10.0
ROTATION_RADIUS_LIMIT = 50.0
COMET_RADIUS = 1.0
COMET_PRODUCTION = 1
PLANET_CLEARANCE = 7
MIN_PLANET_GROUPS = 5
MAX_PLANET_GROUPS = 10
MIN_STATIC_GROUPS = 3
COMET_SPAWN_STEPS = [50, 150, 250, 350, 450]
PLANET_MARGIN = 0.1

CENTER_X = 50.0
CENTER_Y = 50.0
MAX_SPEED = 6.0
MAX_NB_STEP = 500


def distance(p1, p2):
    return math.sqrt((p1[0] - p2[0]) ** 2 + (p1[1] - p2[1]) ** 2)


def point_to_segment_distance(p, v, w):
    """Minimum distance from point p to line segment v-w."""
    l2 = (v[0] - w[0]) ** 2 + (v[1] - w[1]) ** 2
    if l2 == 0.0:
        return distance(p, v)
    t = max(
        0, min(1, ((p[0] - v[0]) * (w[0] - v[0]) + (p[1] - v[1]) * (w[1] - v[1])) / l2)
    )
    projection = (v[0] + t * (w[0] - v[0]), v[1] + t * (w[1] - v[1]))
    return distance(p, projection)

def interpreter(obs, actions, step, num_agents=2):
    # configuration = env.configuration
    obs0 = obs

    # Remove expired comets before fleet launch so agents can't act on them
    expired_comet_pids = []
    for group in obs0.comets:
        idx = group["path_index"]
        for i, pid in enumerate(group["planet_ids"]):
            if idx >= len(group["paths"][i]):
                expired_comet_pids.append(pid)
    if expired_comet_pids:
        expired_set = set(expired_comet_pids)
        obs0.planets = [p for p in obs0.planets if p[0] not in expired_set]
        obs0.initial_planets = [
            p for p in obs0.initial_planets if p[0] not in expired_set
        ]
        obs0.comet_planet_ids = [
            pid for pid in obs0.comet_planet_ids if pid not in expired_set
        ]
        for group in obs0.comets:
            group["planet_ids"] = [
                pid for pid in group["planet_ids"] if pid not in expired_set
            ]
        obs0.comets = [g for g in obs0.comets if g["planet_ids"]]

    # Spawn extra-solar comets at designated steps
    # step = get(obs0, "step", 0)
    # comet_speed = configuration.cometSpeed
    # if (step + 1) in COMET_SPAWN_STEPS:
    #     # Derive a per-spawn RNG from the episode seed so comet shape and
    #     # ship counts are reproducible. Seed lives on env.info to keep it
    #     # hidden from agents (see init block above).
    #     env_info = getattr(env, "info", None) or {}
    #     episode_seed = env_info.get("seed", 0) or 0
    #     comet_rng = random.Random(f"orbit_wars-comet-{episode_seed}-{step + 1}")
    #     comet_paths = generate_comet_paths(
    #         obs0.initial_planets,
    #         obs0.angular_velocity,
    #         step + 1,
    #         obs0.comet_planet_ids,
    #         comet_speed,
    #         rng=comet_rng,
    #     )
    #     if comet_paths:
    #         next_id = max(p[0] for p in obs0.planets) + 1
    #         comet_ships = min(
    #             comet_rng.randint(1, 99),
    #             comet_rng.randint(1, 99),
    #             comet_rng.randint(1, 99),
    #             comet_rng.randint(1, 99),
    #         )
    #         group = {"planet_ids": [], "paths": comet_paths, "path_index": -1}
    #         for i, p_path in enumerate(comet_paths):
    #             pid = next_id + i
    #             group["planet_ids"].append(pid)
    #             obs0.comet_planet_ids.append(pid)
    #             # Start off-board; first advancement will place at path[0]
    #             planet = [
    #                 pid,
    #                 -1,
    #                 -99,
    #                 -99,
    #                 COMET_RADIUS,
    #                 comet_ships,
    #                 COMET_PRODUCTION,
    #             ]
    #             obs0.planets.append(planet)
    #             obs0.initial_planets.append(planet[:])
    #         obs0.comets.append(group)

    # 0. Fleet Launch
    def process_moves(player_id, action):
        if not action or not isinstance(action, list):
            return
        for move in action:
            if len(move) != 3:
                continue
            from_id, angle, ships = move
            ships = int(ships)  # Sanitize to integer

            from_planet = next((p for p in obs0.planets if p[0] == from_id), None)

            if from_planet and from_planet[1] == player_id:
                if from_planet[5] >= ships and ships > 0:
                    from_planet[5] -= ships
                    # Start fleet just outside the planet so it doesn't
                    # immediately collide with its origin.
                    start_x = from_planet[2] + math.cos(angle) * (from_planet[4] + 0.1)
                    start_y = from_planet[3] + math.sin(angle) * (from_planet[4] + 0.1)
                    obs0.fleets.append(
                        [
                            obs0.next_fleet_id,
                            player_id,
                            start_x,
                            start_y,
                            angle,
                            from_id,
                            ships,
                        ]
                    )
                    obs0.next_fleet_id += 1

    for i in range(num_agents):
        process_moves(i, actions[i])

    # 1. Production
    for planet in obs0.planets:
        if planet[1] != -1:
            planet[5] += planet[6]

    # 2. Fleet Movement (with continuous collision detection)
    # Speed scales with fleet size: 1 ship = 1/turn, max = shipSpeed (default 6)
    max_speed = MAX_SPEED
    fleets_to_remove = []
    combat_lists = {p[0]: [] for p in obs0.planets}

    for fleet in obs0.fleets:
        angle = fleet[4]
        ships = fleet[6]
        speed = 1.0 + (max_speed - 1.0) * (math.log(ships) / math.log(1000)) ** 1.5
        speed = min(speed, max_speed)
        old_pos = (fleet[2], fleet[3])
        fleet[2] += math.cos(angle) * speed
        fleet[3] += math.sin(angle) * speed
        new_pos = (fleet[2], fleet[3])

        # Check if fleet path intersected any planet (continuous collision).
        # Check planets first so fast fleets that would overshoot the bounds
        # or sun still get credit for hitting a planet along the way.
        hit_planet = False
        for planet in obs0.planets:
            planet_pos = (planet[2], planet[3])
            if point_to_segment_distance(planet_pos, old_pos, new_pos) < planet[4]:
                combat_lists[planet[0]].append(fleet)
                fleets_to_remove.append(fleet)
                hit_planet = True
                break
        if hit_planet:
            continue

        # Check if fleet went out of bounds
        if not (0 <= fleet[2] <= BOARD_SIZE and 0 <= fleet[3] <= BOARD_SIZE):
            fleets_to_remove.append(fleet)
            continue

        # Check if fleet path crossed the sun
        if point_to_segment_distance((CENTER, CENTER), old_pos, new_pos) < SUN_RADIUS:
            fleets_to_remove.append(fleet)
            continue

    # 3. Planet Movement & Sweep
    angular_velocity = obs0.angular_velocity
    comet_pid_set = set(obs0.comet_planet_ids)
    initial_by_id = {p[0]: p for p in obs0.initial_planets}

    def sweep_fleets(planet, old_pos, new_pos):
        """Check if any fleet is caught by a planet moving from old to new."""
        if old_pos == new_pos:
            return
        for fleet in obs0.fleets:
            if fleet not in fleets_to_remove:
                if (
                    point_to_segment_distance((fleet[2], fleet[3]), old_pos, new_pos)
                    < planet[4]
                ):
                    combat_lists[planet[0]].append(fleet)
                    fleets_to_remove.append(fleet)

    # Regular planet rotation
    for planet in obs0.planets:
        if planet[0] in comet_pid_set:
            continue
        initial_p = initial_by_id.get(planet[0])
        if not initial_p:
            continue
        dx = initial_p[2] - CENTER
        dy = initial_p[3] - CENTER
        r = math.sqrt(dx**2 + dy**2)
        old_pos = (planet[2], planet[3])

        if r + planet[4] < ROTATION_RADIUS_LIMIT:
            initial_angle = math.atan2(dy, dx)
            current_angle = initial_angle + angular_velocity * step
            planet[2] = CENTER + r * math.cos(current_angle)
            planet[3] = CENTER + r * math.sin(current_angle)

        sweep_fleets(planet, old_pos, (planet[2], planet[3]))

    # Comet movement along pre-computed paths
    expired_comet_pids = []
    for group in obs0.comets:
        group["path_index"] += 1
        idx = group["path_index"]
        for i, pid in enumerate(group["planet_ids"]):
            planet = next((p for p in obs0.planets if p[0] == pid), None)
            if planet is None:
                continue
            p_path = group["paths"][i]
            if idx >= len(p_path):
                expired_comet_pids.append(pid)
            else:
                old_pos = (planet[2], planet[3])
                planet[2] = p_path[idx][0]
                planet[3] = p_path[idx][1]
                # Skip sweep on first placement (old_pos is off-board placeholder)
                if old_pos[0] >= 0:
                    sweep_fleets(planet, old_pos, (planet[2], planet[3]))

    # Remove expired comets immediately
    if expired_comet_pids:
        expired_set = set(expired_comet_pids)
        obs0.planets = [p for p in obs0.planets if p[0] not in expired_set]
        obs0.initial_planets = [
            p for p in obs0.initial_planets if p[0] not in expired_set
        ]
        obs0.comet_planet_ids = [
            pid for pid in obs0.comet_planet_ids if pid not in expired_set
        ]
        for group in obs0.comets:
            group["planet_ids"] = [
                pid for pid in group["planet_ids"] if pid not in expired_set
            ]
        obs0.comets = [g for g in obs0.comets if g["planet_ids"]]

    obs0.fleets = [f for f in obs0.fleets if f not in fleets_to_remove]

    # 4. Combat Resolution
    for pid, planet_fleets in combat_lists.items():
        planet = next((p for p in obs0.planets if p[0] == pid), None)
        if not planet or not planet_fleets:
            continue

        # Sum ships per player
        player_ships = {}
        for fleet in planet_fleets:
            owner = fleet[1]
            player_ships[owner] = player_ships.get(owner, 0) + fleet[6]

        if not player_ships:
            continue

        sorted_players = sorted(
            player_ships.items(), key=lambda item: item[1], reverse=True
        )
        top_player, top_ships = sorted_players[0]

        if len(sorted_players) > 1:
            second_ships = sorted_players[1][1]
            survivor_ships = top_ships - second_ships

            if sorted_players[0][1] == sorted_players[1][1]:
                survivor_ships = 0

            survivor_owner = top_player if survivor_ships > 0 else -1
        else:
            survivor_owner = top_player
            survivor_ships = top_ships

        if survivor_ships > 0:
            if planet[1] == survivor_owner:
                planet[5] += survivor_ships
            else:
                planet[5] -= survivor_ships
                if planet[5] < 0:
                    planet[1] = survivor_owner
                    planet[5] = abs(planet[5])

    obs1 = {}
    obs1["planets"] = obs0.planets
    obs1["initial_planets"] = obs0.initial_planets
    obs1["fleets"] = obs0.fleets
    obs1["next_fleet_id"] = obs0.next_fleet_id
    obs1["comets"] = obs0.comets
    obs1["comet_planet_ids"] = obs0.comet_planet_ids

    terminated = False
    if step >= MAX_NB_STEP - 2:
        terminated = True

    alive_players = set()
    for p in obs0.planets:
        if p[1] != -1:
            alive_players.add(p[1])
    for f in obs0.fleets:
        alive_players.add(f[1])

    if len(alive_players) <= 1:
        terminated = True


    return obs1

# Test cases

In [3]:
import copy, math
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

# ── Constants ─────────────────────────────────────────────────────────────────
CENTER = 50.0
SUN_RADIUS = 10.0
ROTATION_RADIUS_LIMIT = 50.0
MAX_SPEED = 6.0
NB_STEPS_SIM = 20
# ---------------------------------------------------------------------------
# Minimal obs object compatible with the interpreter
# ---------------------------------------------------------------------------
class Obs:
    def __init__(self, planets, initial_planets=None, fleets=None,
                 next_fleet_id=100, comets=None, comet_planet_ids=None,
                 angular_velocity=0.0):
        self.planets          = [list(p) for p in planets]
        self.initial_planets  = [list(p) for p in (initial_planets if initial_planets is not None else planets)]
        self.fleets           = [list(f) for f in (fleets or [])]
        self.next_fleet_id    = next_fleet_id
        self.comets           = comets or []
        self.comet_planet_ids = comet_planet_ids or []
        self.angular_velocity = angular_velocity


# Owner -> colour
_COLORS = {0: 'steelblue', 1: 'tomato', -1: '#888888'}


def simulate(obs, n_steps, current_step=0):
    """Step the interpreter n times; return a snapshot list."""
    snapshots = []
    for step in range(current_step, current_step + n_steps):
        snapshots.append({
            'step':    step,
            'planets': [p[:] for p in obs.planets],
            'fleets':  [f[:] for f in obs.fleets],
        })
        interpreter(obs, [[], []], step)
    return snapshots

def simulate_with_action(obs, action0, n_steps, current_step=0):
    """Like simulate() but applies action0 on step 0 only."""
    snapshots = []
    for i, step in enumerate(range(current_step, current_step + n_steps)):
        snapshots.append({
            'step':    step,
            'planets': [p[:] for p in obs.planets],
            'fleets':  [f[:] for f in obs.fleets],
        })
        interpreter(obs, [action0 if i == 0 else [], []], step)
    return snapshots


def _simulate(obs, global_step, num_agents, n_steps=NB_STEPS_SIM):
    sim = copy.deepcopy(obs)
    no_actions = [[] for _ in range(num_agents)]
    rows = []
    for i in range(n_steps+1):
        for p in sim.planets:
            pid, owner, x, y, radius, ships, production = (
                p[0], p[1], p[2], p[3], p[4], p[5], p[6]
            )
            r = math.hypot(x - CENTER, y - CENTER)
            if pid in sim.comet_planet_ids:
                nature = "comet"
            elif r + radius < ROTATION_RADIUS_LIMIT:
                nature = "moving"
            else:
                nature = "fix"
            rows.append({
                "step": global_step + i,
                "id": pid,
                "x": x,
                "y": y,
                "radius": radius,
                "ships": ships,
                "production": production,
                "owner": owner,
                "nature": nature,
            })
        interpreter(sim, no_actions, global_step + i, num_agents)
    return pd.DataFrame(rows)


def make_animation(snapshots, title='', interval=150):
    """Animate a snapshot list produced by simulate()."""
    fig, ax = plt.subplots(figsize=(6, 6))
    fig.patch.set_facecolor('#111122')

    def draw(frame):
        snap = snapshots[frame]
        ax.cla()
        ax.set_xlim(0, 100)
        ax.set_ylim(100, 0)   # y decreases downward: 100 at top, 0 at bottom
        ax.set_aspect('equal')
        ax.set_facecolor('#111122')
        ax.tick_params(colors='#aaaaaa')
        for sp in ax.spines.values():
            sp.set_edgecolor('#444444')
        ax.set_title(f"{title}  (step {snap['step']})", color='white', fontsize=11)

        # Sun
        ax.add_patch(plt.Circle((50, 50), 10, color='gold', zorder=2, alpha=0.9))

        # Planets
        for p in snap['planets']:
            pid, owner, x, y, radius, ships, production = p
            c = _COLORS.get(owner, '#888888')
            ax.add_patch(plt.Circle((x, y), radius, color=c, alpha=0.85, zorder=3))
            ax.text(x, y, str(ships),
                    ha='center', va='center', color='white',
                    fontsize=7, fontweight='bold', zorder=4)

        # Fleets
        for f in snap['fleets']:
            fid, owner, x, y, angle, from_id, ships = f
            c = _COLORS.get(owner, '#888888')
            ax.plot(x, y, 'D', color=c, markersize=5, zorder=5)
            ax.text(x + 1.5, y + 1.5, str(ships), color=c, fontsize=5, zorder=6)

        return []

    ani = animation.FuncAnimation(fig, draw, frames=len(snapshots), interval=interval)
    plt.close()
    return HTML(ani.to_jshtml())

In [4]:
import numpy as np

def take_action(df, player_id, nb_steps_sim=NB_STEPS_SIM):
    mine_across_sim = (
        df
        .query("owner == @player_id")
        .groupby("id")
        .agg(
            step_src=("step", "first"),
            x_src=("x", "first"),
            y_src=("y", "first"),
            radius_src=("radius", "first"),
            ships_min=("ships", "min"),
            production_src=("production", "first"),
            nature_src=("nature", "first"),
            row_count=("ships", "size"),
        )
        .query("row_count >= @nb_steps_sim + 1")
        .reset_index(drop=False)
        .rename(columns={"id": "id_src"})
    )

    possible_attacks = (
        mine_across_sim
        .merge(
            df,
            how="cross"
        )
        .query("step > step_src and id != id_src")
        .assign(
            dist_tgt_src=lambda d: ((d["x"] - d["x_src"]) ** 2 + (d["y"] - d["y_src"]) ** 2) ** 0.5 - d["radius_src"] - d["radius"] - PLANET_MARGIN,
            step_diff=lambda d: d["step"] - d["step_src"], # For a one ship fleet, the speed is 1 unit per step
            ships_needed=lambda d: d["ships"] + 1, # Need at least one more ship than the target to win
            fleet_speed=lambda d: 1.0 + (MAX_SPEED - 1.0) * ((np.log(d["ships_needed"])) / math.log(1000)) ** 1.5,
            possible_attack=lambda d: (d["dist_tgt_src"] / d["fleet_speed"] <= d["step_diff"]),
        )
        .query("possible_attack and ships_min >= ships_needed")
        .sort_values(["step_src", "id_src", "step", "id"], ascending=[True, True, True, True])
        .groupby(["step_src", "id_src", "id"], as_index=False)
        .first()
        .assign(
            angle=lambda d: np.arctan2(d["y"] - d["y_src"], d["x"] - d["x_src"]),
        )
        .sort_values("step", ascending=True)
    )
    moves = (
        possible_attacks
        [["id_src", "angle", "ships_needed"]]
        .values
        .tolist()
    )
    return moves

## Test 1 — Planet production
One static planet with `production=3`. Ships should increase by 3 every step.

In [5]:
# planet: id=0, owner=0, x=10, y=10, radius=5, ships=1, production=3
obs1 = Obs(
    planets=[[0, 0, 10.0, 10.0, 5.0, 1, 3]],
    angular_velocity=0.0,
)
df = _simulate(obs1, global_step=0, num_agents=2, n_steps=NB_STEPS_SIM)
print(df)
snaps1 = simulate(obs1, 10, current_step=0)
make_animation(snaps1, title='Test 1 — Planet Production (prod=3)', interval=400)

    step  id     x     y  radius  ships  production  owner nature
0      0   0  10.0  10.0     5.0      1           3      0    fix
1      1   0  10.0  10.0     5.0      4           3      0    fix
2      2   0  10.0  10.0     5.0      7           3      0    fix
3      3   0  10.0  10.0     5.0     10           3      0    fix
4      4   0  10.0  10.0     5.0     13           3      0    fix
5      5   0  10.0  10.0     5.0     16           3      0    fix
6      6   0  10.0  10.0     5.0     19           3      0    fix
7      7   0  10.0  10.0     5.0     22           3      0    fix
8      8   0  10.0  10.0     5.0     25           3      0    fix
9      9   0  10.0  10.0     5.0     28           3      0    fix
10    10   0  10.0  10.0     5.0     31           3      0    fix
11    11   0  10.0  10.0     5.0     34           3      0    fix
12    12   0  10.0  10.0     5.0     37           3      0    fix
13    13   0  10.0  10.0     5.0     40           3      0    fix
14    14  

## Test 2 — Attacking Neutral
Player 0 (6 ships) vs neutral (5 ships) — player 0 wins outright → should attack.

In [6]:
import copy

obs1 = Obs(
    planets=[[0, 0, 10.0, 10.0, 5.0, 6, 3], [1, -1, 30.0, 10.0, 5.0, 5, 2]],
    angular_velocity=0.0,
)

df2 = _simulate(obs1, global_step=0, num_agents=2, n_steps=NB_STEPS_SIM)
print(df2)
# Player 0 attacks: 6 ships > 5 defender → clear win
action = take_action(df2, player_id=0, nb_steps_sim=NB_STEPS_SIM)
print(f"\nAction: {action}")
snaps2 = simulate_with_action(copy.deepcopy(obs1), action, 10)
make_animation(snaps2, title='Test 2 — Attacking Neutral', interval=200)

    step  id     x     y  radius  ships  production  owner  nature
0      0   0  10.0  10.0     5.0      6           3      0     fix
1      0   1  30.0  10.0     5.0      5           2     -1  moving
2      1   0  10.0  10.0     5.0      9           3      0     fix
3      1   1  30.0  10.0     5.0      5           2     -1  moving
4      2   0  10.0  10.0     5.0     12           3      0     fix
5      2   1  30.0  10.0     5.0      5           2     -1  moving
6      3   0  10.0  10.0     5.0     15           3      0     fix
7      3   1  30.0  10.0     5.0      5           2     -1  moving
8      4   0  10.0  10.0     5.0     18           3      0     fix
9      4   1  30.0  10.0     5.0      5           2     -1  moving
10     5   0  10.0  10.0     5.0     21           3      0     fix
11     5   1  30.0  10.0     5.0      5           2     -1  moving
12     6   0  10.0  10.0     5.0     24           3      0     fix
13     6   1  30.0  10.0     5.0      5           2     -1  mo

## Test 3 — Do nothing
Player 0 (5 ships) vs neutral (5 ships) — tie destroys both fleets → should not attack.

In [7]:
obs1 = Obs(
    planets=[[0, 0, 10.0, 10.0, 5.0, 5, 3], [1, -1, 30.0, 10.0, 5.0, 5, 2]],
    angular_velocity=0.0,
)
df3 = _simulate(obs1, global_step=0, num_agents=2, n_steps=5)
print(df3)
action = take_action(df3, player_id=0, nb_steps_sim=5)
print(f"\nAction: {action} (do nothing — equal ships, tie loses)")
snaps3 = simulate_with_action(copy.deepcopy(obs1), action, 10)
make_animation(snaps3, title='Test 3 — Do Nothing (equal ships)', interval=200)

    step  id     x     y  radius  ships  production  owner  nature
0      0   0  10.0  10.0     5.0      5           3      0     fix
1      0   1  30.0  10.0     5.0      5           2     -1  moving
2      1   0  10.0  10.0     5.0      8           3      0     fix
3      1   1  30.0  10.0     5.0      5           2     -1  moving
4      2   0  10.0  10.0     5.0     11           3      0     fix
5      2   1  30.0  10.0     5.0      5           2     -1  moving
6      3   0  10.0  10.0     5.0     14           3      0     fix
7      3   1  30.0  10.0     5.0      5           2     -1  moving
8      4   0  10.0  10.0     5.0     17           3      0     fix
9      4   1  30.0  10.0     5.0      5           2     -1  moving
10     5   0  10.0  10.0     5.0     20           3      0     fix
11     5   1  30.0  10.0     5.0      5           2     -1  moving

Action: [] (do nothing — equal ships, tie loses)


## Test 4 — Do not attack if fleet arriving
Player 0 (5 ships) vs neutral (5 ships) but an enemy fleet (50 ships) is inbound → wasting ships on a planet the enemy will take anyway.

In [8]:
obs1 = Obs(
    planets=[[0, 0, 10.0, 10.0, 5.0, 6, 3], [1, -1, 30.0, 10.0, 5.0, 5, 2]],
    fleets=[[0, 1, 10.0, 30.0, 3 * math.pi / 2, 0, 24]],
    next_fleet_id=1,
    angular_velocity=0.0,
)
df4 = _simulate(obs1, global_step=0, num_agents=2, n_steps=10)
print(df4)
action = take_action(df4, player_id=0, nb_steps_sim=10)
print(f"\nAction: {action} (do nothing — enemy fleet inbound)")
snaps4 = simulate_with_action(copy.deepcopy(obs1), action, 15)
make_animation(snaps4, title='Test 4 — Do Not Attack (fleet arriving)', interval=200)

    step  id     x     y  radius  ships  production  owner  nature
0      0   0  10.0  10.0     5.0      6           3      0     fix
1      0   1  30.0  10.0     5.0      5           2     -1  moving
2      1   0  10.0  10.0     5.0      9           3      0     fix
3      1   1  30.0  10.0     5.0      5           2     -1  moving
4      2   0  10.0  10.0     5.0     12           3      0     fix
5      2   1  30.0  10.0     5.0      5           2     -1  moving
6      3   0  10.0  10.0     5.0     15           3      0     fix
7      3   1  30.0  10.0     5.0      5           2     -1  moving
8      4   0  10.0  10.0     5.0     18           3      0     fix
9      4   1  30.0  10.0     5.0      5           2     -1  moving
10     5   0  10.0  10.0     5.0     21           3      0     fix
11     5   1  30.0  10.0     5.0      5           2     -1  moving
12     6   0  10.0  10.0     5.0      0           3      0     fix
13     6   1  30.0  10.0     5.0      5           2     -1  mo

## Test 5 — Attacking Enemy properly
Player 0 (50 ships) vs neutral (5 ships) — overwhelming advantage → should attack.

In [9]:
obs1 = Obs(
    planets=[[0, 0, 10.0, 10.0, 5.0, 50, 3], [1, 1, 30.0, 10.0, 5.0, 5, 2]],
    angular_velocity=0.0,
)
df5 = _simulate(obs1, global_step=0, num_agents=2, n_steps=5)
print(df5)  
action = take_action(df5, player_id=0, nb_steps_sim=5)
print(f"\nAction: {action} (attack with overwhelming force)")
snaps5 = simulate_with_action(copy.deepcopy(obs1), action, 10)
make_animation(snaps5, title='Test 5 — Attacking Enemy Properly', interval=200)

    step  id     x     y  radius  ships  production  owner  nature
0      0   0  10.0  10.0     5.0     50           3      0     fix
1      0   1  30.0  10.0     5.0      5           2      1  moving
2      1   0  10.0  10.0     5.0     53           3      0     fix
3      1   1  30.0  10.0     5.0      7           2      1  moving
4      2   0  10.0  10.0     5.0     56           3      0     fix
5      2   1  30.0  10.0     5.0      9           2      1  moving
6      3   0  10.0  10.0     5.0     59           3      0     fix
7      3   1  30.0  10.0     5.0     11           2      1  moving
8      4   0  10.0  10.0     5.0     62           3      0     fix
9      4   1  30.0  10.0     5.0     13           2      1  moving
10     5   0  10.0  10.0     5.0     65           3      0     fix
11     5   1  30.0  10.0     5.0     15           2      1  moving

Action: [[0.0, 0.0, 16.0]] (attack with overwhelming force)


## Test 6 — Our Agent vs Random
Run `36-Dataframe_nearest_10steps.py` (player 0, blue) against a random agent (player 1, red) using the Kaggle environment. Module is reloaded each run to reset global step/player state.

In [11]:
# import kaggle_environments as ke
import random, math

SEED = 42
N_STEPS = 100
random.seed(SEED)


def random_agent_fn(obs):
    player = obs.player
    my_planets = [p for p in obs.planets if p[1] == player]
    if not my_planets:
        return []
    planet = random.choice(my_planets)
    ships = planet[5] // 2
    if ships < 1:
        return []
    return [[planet[0], random.uniform(0, 2 * math.pi), ships]]

env = ke.make("orbit_wars", debug=False)
env.reset(2)

snaps6 = []
for env_step in range(N_STEPS):
    obs0 = env.state[0].observation
    obs1 = env.state[1].observation
    snaps6.append({
        'step':    env_step,
        'planets': [list(p) for p in obs0.planets],
        'fleets':  [list(f) for f in obs0.fleets],
    })
    df = _simulate(obs0, global_step=env_step, num_agents=2, n_steps=NB_STEPS_SIM)
    action0 = take_action(df, player_id=0, nb_steps_sim=NB_STEPS_SIM)
    action1 = random_agent_fn(obs1)
    env.step([action0, action1])
    if env.state[0].status != "ACTIVE":
        break

# Capture final state
obs0 = env.state[0].observation
snaps6.append({
    'step':    len(snaps6),
    'planets': [list(p) for p in obs0.planets],
    'fleets':  [list(f) for f in obs0.fleets],
})

p0_ships = sum(p[5] for p in obs0.planets if p[1] == 0)
p1_ships = sum(p[5] for p in obs0.planets if p[1] == 1)
print(f"Player 0 (our agent): {p0_ships} ships on planets")
print(f"Player 1 (random):    {p1_ships} ships on planets")
winner = "Our agent wins" if p0_ships > p1_ships else "Random wins" if p1_ships > p0_ships else "Tie"
print(f"After {len(snaps6) - 1} steps: {winner}")

make_animation(snaps6, title='Test 6 — Our Agent vs Random', interval=100)

Player 0 (our agent): 529 ships on planets
Player 1 (random):    3 ships on planets
After 100 steps: Our agent wins


In [12]:
(0.6**2 + 0.9 **2) ** 0.5

1.0816653826391966